# Ch.3 — Feature Scaling, Importance & Multicollinearity

**Goal:** SmartVal AI has 8 features and a $55k MAE from Ch.2. Before adding polynomial complexity (Ch.4) or regularisation (Ch.5), we need to answer:
- Why must we scale features before comparing their importance?
- Which features carry the most signal toward house value?
- Which features measure the same thing (and therefore compete for the same weights)?
- What does each diagnostic method reveal that the others miss?

By the end of this notebook you will have a **three-view feature dashboard** and specific action items for Ch.4 and Ch.5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# Deterministic results
np.random.seed(42)

print('Libraries loaded ✓')

## 0 · Feature Scaling

Before any importance ranking is possible, all features must be on a **common scale**. Raw weights from an unstandardized model are not comparable — a weight of 200 for `Population` (range 3–35,682) looks enormous next to a weight of 0.4 for `MedInc` (range 0.5–15), but that gap reflects the input scale, not the feature's importance.

**Standardization (Z-score):** $x_j^{\text{std}} = \dfrac{x_j - \mu_j}{\sigma_j}$

After standardization every feature has mean = 0, std = 1, so weight magnitudes are directly comparable.

> ⚠️ **Pipeline rule:** Always fit the scaler on **training data only**, then `transform` both train and test. Fitting on the full dataset leaks test statistics into training.

In [ ]:
housing = fetch_california_housing()
X_raw = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Fit scaler on TRAIN only, then transform both splits
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_raw)
X_test_s  = scaler.transform(X_test_raw)   # no fit here — avoids data leakage

# ── Before vs After comparison ──────────────────────────────────────────────
print("Feature ranges BEFORE standardization:")
print(X_train_raw[['MedInc', 'Population', 'AveRooms', 'Latitude']].agg(['min', 'max']).round(1).to_string())

X_train_df = pd.DataFrame(X_train_s, columns=housing.feature_names)
print("\nFeature ranges AFTER standardization:")
print(X_train_df[['MedInc', 'Population', 'AveRooms', 'Latitude']].agg(['min', 'max']).round(2).to_string())

# ── Visualise the effect on gradient magnitudes ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Feature Scaling: Before vs After", fontsize=13, fontweight='bold')

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12',
          '#9b59b6', '#1abc9c', '#e67e22', '#34495e']

for ax, (X_df, title) in zip(axes, [
    (X_train_raw, 'BEFORE StandardScaler\n(raw ranges)'),
    (X_train_df,  'AFTER StandardScaler\n(mean=0, std=1)'),
]):
    ranges = X_df.max() - X_df.min()
    bars = ax.bar(housing.feature_names, ranges, color=colors)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Range (max − min)')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, ranges):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02, f'{val:.1f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\n✅ Train samples: {X_train_s.shape[0]:,}  |  Test samples: {X_test_s.shape[0]:,}")
print("Scaler fitted on training data only — no test leakage.")

## 1 · Fit the Baseline Model

Data is already loaded and scaled from Section 0. We just fit the Ch.2 LinearRegression on the standardized splits.

In [ ]:
# X_train_s / X_test_s / y_train / y_test already produced in Section 0
X = X_train_raw  # alias — full feature DataFrame for labelling elsewhere

# Fit Ch.2 model on standardized features
model = LinearRegression()
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
mae = mean_absolute_error(y_test, y_pred) * 100_000  # convert to dollars
print(f'Ch.2 baseline MAE: ${mae:,.0f}')
print(f'Number of features: {X_train_s.shape[1]}')
print(f'Training samples:   {X_train_s.shape[0]:,}')

## 2 · Method 1 — Univariate R²

**Question:** If I used only this feature, how much target variance would it explain?

**Shortcut:** For linear regression, univariate R² = ρ(xⱼ, y)². Read it from the correlation matrix — no need to fit 8 models.

In [ ]:
# Build a combined DataFrame with the target
df_train = X_train.copy()
df_train['MedHouseVal'] = y_train

corr_with_target = df_train.corr()['MedHouseVal'].drop('MedHouseVal')
univariate_r2 = (corr_with_target ** 2).sort_values(ascending=False)

print('Univariate R² (ρ² shortcut):')
print('-' * 40)
for feat, val in univariate_r2.items():
    bar = '█' * int(val * 80)
    print(f'  {feat:12s}  {val:.3f}  {bar}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1d4ed8' if v > 0.05 else '#94a3b8' for v in univariate_r2.values]
bars = ax.barh(univariate_r2.index[::-1], univariate_r2.values[::-1], color=colors[::-1])
for bar, val in zip(bars, univariate_r2.values[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Univariate R²  (proportion of target variance explained alone)')
ax.set_title('Feature Importance — Method 1: Univariate R²\n'
             '"If I used only this feature, how much would I explain?"', fontsize=11)
ax.axvline(0.05, color='#ef4444', linestyle='--', alpha=0.6, label='5% threshold')
ax.legend(fontsize=9)
ax.set_xlim(0, 0.55)
plt.tight_layout()
plt.savefig('img/univariate_r2.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nObservation: MedInc towers over everything. Every other feature looks flat at this scale.')
print('This is NOT the full picture — it is the starting question.')

## 3 · Method 2 — Standardised Weights (Partial Contribution)

**Question:** Given all other features in the model, how much does each feature contribute?

This is the *partial* effect — what each feature adds above and beyond everything else.

In [ ]:
std_weights = pd.Series(
    model.coef_,
    index=housing.feature_names
)
abs_weights = std_weights.abs().sort_values(ascending=False)

print('Standardised weights (|wⱼ|) — partial contribution:')
print('-' * 50)
for feat, val in abs_weights.items():
    sign = '+' if std_weights[feat] >= 0 else '−'
    bar = '█' * int(val * 10)
    print(f'  {feat:12s}  {sign}{val:.3f}  {bar}')

print('\nNote: Latitude and Longitude are now at the TOP — they were near the BOTTOM in univariate R².')
print('This reversal is the key teaching moment.')

## 4 · The Surprise — Why the Rankings Diverge

Let's directly visualise the ranking gap between the two methods.

In [ ]:
# Normalise both to 0-1 scale for visual comparison
uni_norm = univariate_r2 / univariate_r2.max()
wt_norm  = abs_weights   / abs_weights.max()

comparison = pd.DataFrame({
    'Univariate R² (alone)':      uni_norm,
    'Std weight (joint model)': wt_norm
}).sort_values('Std weight (joint model)', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(comparison))
w = 0.35
ax.bar(x - w/2, comparison['Univariate R² (alone)'],      w, label='Univariate R² (alone)',    color='#94a3b8', alpha=0.85)
ax.bar(x + w/2, comparison['Std weight (joint model)'], w, label='Std weight (joint)',  color='#1d4ed8', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=20, ha='right')
ax.set_ylabel('Normalised importance (1 = max)')
ax.set_title('Where the Two Methods Agree — and Where They Diverge\n'
             'Blue bars rising vs grey: jointly irreplaceable (Lat, Lon)\n'
             'Grey bars falling vs blue: signal is shared with others (AveRooms)', fontsize=10)
ax.legend()
plt.tight_layout()
plt.savefig('img/ranking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey divergences:')
print('  Lat/Lon:   low univariate → high joint  → JOINTLY IRREPLACEABLE')
print('  AveRooms:  moderate univariate → lower joint → SIGNAL SHARED with AveBedrms')

## 5 · Feature Correlation Heatmap

Visualising *feature × feature* correlations reveals the collinear pairs **before** computing VIF.

In [ ]:
corr_matrix = X_train.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle only
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Heatmap\n'
             'High |ρ| between features = collinearity risk\n'
             'Red circles: AveRooms/AveBedrms (ρ≈0.85) and Lat/Lon (ρ≈−0.92)', fontsize=10)
# Highlight the two collinear pairs with rectangles
feat_names = list(corr_matrix.columns)
for pair in [('AveRooms', 'AveBedrms'), ('Latitude', 'Longitude')]:
    i, j = feat_names.index(pair[0]), feat_names.index(pair[1])
    for (r, c) in [(i, j), (j, i)]:
        ax.add_patch(plt.Rectangle((c, r), 1, 1, fill=False,
                                    edgecolor='black', lw=2.5))
plt.tight_layout()
plt.savefig('img/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AveRooms / AveBedrms correlation: {corr_matrix.loc['AveRooms','AveBedrms']:.2f}")
print(f"Latitude / Longitude  correlation: {corr_matrix.loc['Latitude','Longitude']:.2f}")

## 6 · VIF — Quantifying Multicollinearity

VIF measures how much each feature's weight is inflated by its correlation with the **other features**.

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the R² from regressing feature $j$ on **all other features** (not the target).

In [ ]:
vif_df = pd.DataFrame({
    'Feature': housing.feature_names,
    'VIF': [variance_inflation_factor(X_train.values, i)
            for i in range(X_train.shape[1])]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

def vif_flag(v):
    if v > 10: return '❌  SEVERE'
    if v > 5:  return '⚡  HIGH'
    if v > 2:  return '⚠️   Moderate'
    return '✅  OK'

vif_df['Status'] = vif_df['VIF'].apply(vif_flag)
print('VIF Analysis:')
print(vif_df.to_string(index=False))

print('\nInterpretation:')
print('  VIF = 7.2 on AveRooms means its standard error is √7.2 ≈ 2.7× larger than')
print('  it would be if AveRooms were uncorrelated with all other features.')
print('  The weight estimate is noisy — not necessarily wrong, but unreliable for interpretation.')

In [ ]:
# Demonstrate weight instability under multicollinearity
print('Demonstrating weight instability (AveRooms vs AveBedrms):')
print('──────────────────────────────────────────────────────────')
for seed in [42, 7, 99, 1234, 2025]:
    _, X_sub, _, y_sub = train_test_split(
        X_train, y_train, test_size=0.5, random_state=seed
    )
    scaler_sub = StandardScaler()
    X_sub_s = scaler_sub.fit_transform(X_sub)
    m = LinearRegression().fit(X_sub_s, y_sub)
    idx_rooms    = housing.feature_names.tolist().index('AveRooms')
    idx_bedrms   = housing.feature_names.tolist().index('AveBedrms')
    w_rooms  = m.coef_[idx_rooms]
    w_bedrms = m.coef_[idx_bedrms]
    print(f'  seed={seed:4d}  AveRooms={w_rooms:+.3f}  AveBedrms={w_bedrms:+.3f}')

print('\nThe weights flip sign and magnitude across subsets.')
print('Predictions on the full test set are nearly identical despite this — VIF in action.')

## 7 · Method 3 — Permutation Importance

**Question:** If I *scramble* this feature on the test set (destroying its signal), how much does MAE rise?

This is the **most reliable** and model-agnostic method. It directly measures the model's dependence on each feature.

In [ ]:
perm = permutation_importance(
    model, X_test_s, y_test,
    n_repeats=30,
    scoring='neg_mean_absolute_error',
    random_state=42
)
perm_imp = pd.Series(
    perm.importances_mean, index=housing.feature_names
).sort_values(ascending=False)
perm_std = pd.Series(perm.importances_std, index=housing.feature_names)

print('Permutation importance (mean Δ MAE when feature is shuffled):')
print('-' * 60)
for feat in perm_imp.index:
    val = perm_imp[feat]
    std = perm_std[feat]
    delta_usd = val * 100_000
    print(f'  {feat:12s}  {val:.4f} ± {std:.4f}  (≈${delta_usd:+,.0f} MAE rise if shuffled)')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1d4ed8' if v > 0.01 else '#94a3b8' for v in perm_imp.values]
ax.barh(
    perm_imp.index[::-1], perm_imp.values[::-1],
    xerr=perm_std[perm_imp.index[::-1]].values,
    color=colors[::-1], alpha=0.85, capsize=3
)
ax.set_xlabel('Mean Δ MAE (neg_mean_absolute_error) when feature is shuffled')
ax.set_title('Feature Importance — Method 3: Permutation Importance\n'
             'Error bars = std across 30 repeats', fontsize=11)
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('img/permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Three-View Dashboard

Side-by-side comparison of all three methods. Features where rankings diverge tell the richest story.

In [ ]:
dashboard = pd.DataFrame({
    'Univariate R²':    univariate_r2,
    '|Std weight|':     abs_weights,
    'Perm importance':  perm_imp
})

# Rank each column (1 = most important)
ranks = dashboard.rank(ascending=False).astype(int)
ranks.columns = ['Rank (uni R²)', 'Rank (std wt)', 'Rank (perm)']

full_dash = pd.concat([dashboard.round(3), ranks], axis=1)

def verdict(row):
    r = (row['Rank (uni R²)'], row['Rank (std wt)'], row['Rank (perm)'])
    if max(r) <= 3:
        return '✅ Strong independent signal'
    if r[0] >= 5 and r[1] <= 3:
        return '🔵 Jointly irreplaceable'
    if row['Perm importance'] < 0.005:
        return '❌ Near-zero contribution'
    if r[1] >= 6 and row['|Std weight|'] < 0.05:
        return '⚡ Collinear — weight unstable'
    return '⚠️  Modest'

full_dash['Verdict'] = full_dash.apply(verdict, axis=1)
print(full_dash.to_string())

In [ ]:
# Visual three-panel summary
fig = plt.figure(figsize=(15, 4.5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.45)

methods = [
    ('Univariate R²\n(alone)', univariate_r2, '#94a3b8'),
    ('Std |weight|\n(joint)', abs_weights, '#1d4ed8'),
    ('Permutation\n(test set)', perm_imp, '#15803d'),
]
for idx, (title, data, color) in enumerate(methods):
    ax = fig.add_subplot(gs[0, idx])
    order = data.sort_values(ascending=True)
    ax.barh(order.index, order.values, color=color, alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Importance')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

fig.suptitle('Three-View Feature Importance Dashboard — California Housing',
             fontsize=12, fontweight='bold', y=1.02)
plt.savefig('img/three_view_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: img/three_view_dashboard.png')

## 9 · Multicollinearity Deep Dive — AveRooms vs AveBedrms

Visualise *why* the two room features cause weight instability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: AveRooms vs AveBedrms
axes[0].scatter(
    X_train['AveRooms'].clip(0, 10),
    X_train['AveBedrms'].clip(0, 5),
    alpha=0.05, s=4, color='#1d4ed8'
)
rho = X_train['AveRooms'].corr(X_train['AveBedrms'])
axes[0].set_xlabel('AveRooms (clipped at 10)')
axes[0].set_ylabel('AveBedrms (clipped at 5)')
axes[0].set_title(f'AveRooms vs AveBedrms\nρ = {rho:.2f}  (highly collinear)\n'
                  'The model cannot cleanly separate their contributions.', fontsize=9)

# Weight instability across bootstrap sub-samples
rooms_wts, bedrms_wts = [], []
idx_r = housing.feature_names.tolist().index('AveRooms')
idx_b = housing.feature_names.tolist().index('AveBedrms')
for seed in range(100):
    _, X_sub, _, y_sub = train_test_split(
        X_train, y_train, test_size=0.3, random_state=seed
    )
    sc = StandardScaler()
    Xs = sc.fit_transform(X_sub)
    c  = LinearRegression().fit(Xs, y_sub).coef_
    rooms_wts.append(c[idx_r])
    bedrms_wts.append(c[idx_b])

axes[1].scatter(rooms_wts, bedrms_wts, alpha=0.5, s=20, color='#ef4444')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('AveRooms weight (100 bootstrap sub-samples)')
axes[1].set_ylabel('AveBedrms weight')
axes[1].set_title('Weight instability across sub-samples\n'
                  'Each dot = one 70% sub-sample. Scatter = unstable estimates.\n'
                  'Predictions are fine; individual weights are not.', fontsize=9)

plt.tight_layout()
plt.savefig('img/collinearity_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 · Action Items for Ch.4 & Ch.5

Based on the three-view dashboard, what do we know about pursuing the $40k MAE target?

In [ ]:
print('═' * 65)
print(' FEATURE IMPORTANCE AUDIT — ACTION ITEMS')
print('═' * 65)

print('''
Ch.4 · Polynomial Features (target: ~$48k MAE)
───────────────────────────────────────────────
✅ HIGH-VALUE polynomial targets (strong permutation importance):
   • MedInc²  — income has a known non-linear ceiling effect
   • MedInc × Latitude  — coastal premium (geographic income signal)
   • MedInc × Longitude — East/West income gradient
   • Latitude × Longitude — geographic cluster interactions

⚠️  CAUTIOUS additions (low permutation importance):
   • AveBedrms² — very low permutation importance; adds noise risk
   • Population² — near-zero contribution; don't expand

Ch.5 · Regularization (target: ~$38k MAE)
──────────────────────────────────────────
⚡ Collinear pair to handle with Ridge:
   • AveRooms / AveBedrms (VIF ≈ 7)  → Ridge shrinks toward shared signal
   • Alternative: drop AveBedrms, or create rooms_per_bedroom feature

⚡ Correlated pair (informative but anti-correlated):
   • Latitude / Longitude (VIF ≈ 3.5) → keep both; they are jointly irreplaceable
   • Ridge safely handles their moderate VIF

❌ Feature to potentially drop:
   • Population — permutation importance ≈ 0; adds noise, costs a parameter
''')

print('═' * 65)

## Summary

| Method | Top feature | Bottom feature | Best use |
|---|---|---|---|
| Univariate R² | MedInc (0.47) | Population (0.001) | First-pass scan; quick ranking |
| Std \|weight\| | Latitude (0.89) | Population (0.01) | Final model inspection; requires standardisation |
| Permutation | MedInc (+$18k) | Population (+$0.1k) | Most reliable; model-agnostic; use on test set |

**Key takeaways:**
1. **MedInc** is the strongest individual predictor regardless of method.
2. **Latitude and Longitude** are jointly irreplaceable — low alone, high together.
3. **AveRooms / AveBedrms** are collinear (VIF ≈ 7) — their weight split is arbitrary.
4. **Population** contributes virtually nothing and is a candidate for removal.
5. Always inspect all three views; a feature cheap in one can be critical in another.

**Next:** Ch.4 — Polynomial Features: add `MedInc²`, `MedInc × Latitude`, `MedInc × Longitude` to push MAE toward $48k.